# Nested Learning for CAD -- recovery stage 1: controls

Worktree `hope_cad_v5`; algorithms remain **frozen v4**. No W6 refit, learner revision, full matrix, or VisA. CADIC uses an explicitly identified exact tile-min cache with reference parity.

**r2: do not attach the FAILED recovery output; start a fresh preflight with these 5 inputs.**

**Preflight inputs (5):** NEW recovery source bundle; completed W6 v4; completed W7.2 from `w7_kaggle_7_2_1`; MVTec AD; ViT-B/8 checkpoint.
**Controls/resume inputs (5):** same source/W6/MVTec/ViT; replace W7.2 input with the LATEST recovery Output/ZIP (preflight first).
You may leave reviewed W7.2 attached; recovered parent copies are verified. Detach older recovery outputs; keep only latest.

Use **GPU T4 x2**, Internet not required, **Save Version -> Save & Run All**. Runner uses one GPU.
Preflight runs regressions, normal RGB parity, and a **600-second bounded** cached K=10000 CADIC probe including full-bank reference parity. The cache is extra compute memory, disclosed separately. If it times out, preserve ZIP and stop for engineering review; do not disable the gate.
Controls run C1 -> A1 -> CADIC10k (operational order only), with checkpoints and final metrics. Normal diagnostics read M's saved states and new C1/A1 states. No full W7.3.
A paused output resumes the SAME study from saved boundaries. Completion needs artifact review. A hard platform kill can prevent final export; normal pauses/failures export before raising.


In [ ]:
PHASE='controls'

import os,sys,time,json,hashlib,zipfile,shutil
from pathlib import Path,PurePosixPath
STARTED=time.monotonic()
sys.dont_write_bytecode=True
os.environ['PYTHONDONTWRITEBYTECODE']='1'
INPUT=Path('/kaggle/input')
REPO=Path('/tmp/hope_recovery_stage1_source')
WORK=Path('/kaggle/working/hope_recovery_stage1')
W6=Path('/tmp/hope_recovery_parent_w6')
ASSETS=Path('/tmp/hope_recovery_assets')
OUTPUT=Path('/kaggle/working/hope_cad_recovery_stage1_evidence.zip')
SOURCE_OVERRIDE=None
W6_OVERRIDE=None
W72_OVERRIDE=None
RESUME_OVERRIDE=None
MVTEC_OVERRIDE=None
CHECKPOINT_OVERRIDE=None
SESSION_HOURS=10.5
JOBS=['C1_seed0','A1_seed0','CADIC10000_seed0']

def sha(p):
    with Path(p).open('rb') as f:return hashlib.file_digest(f,'sha256').hexdigest()
def safe(root,name):
    rel=PurePosixPath(name)
    path=(root/name).resolve()
    if rel.is_absolute() or '..' in rel.parts or rel.as_posix()!=name or chr(92) in name or ':' in name or not path.is_relative_to(root.resolve()):
        raise ValueError('Unsafe source path: '+name)
    return path
print('[BOOTSTRAP] locating recovery source; elapsed',round(time.monotonic()-STARTED,1),'s',flush=True)
sources=[]
for p in INPUT.rglob('bundle_manifest.json'):
    try:
        if json.loads(p.read_text()).get('schema')=='hope_recovery_stage1_source_v1':sources.append(p.parent)
    except ValueError:pass
if not sources:sources=list(INPUT.rglob('hope_cad_recovery_stage1_source*.zip'))
if SOURCE_OVERRIDE:source=Path(SOURCE_OVERRIDE)
else:
    if not sources:raise ValueError('Attach the NEW hope_cad_recovery_stage1_source bundle')
    manifests=[]
    for p in sorted(set(sources)):
        if p.is_dir():raw=(p/'bundle_manifest.json').read_bytes()
        else:
            with zipfile.ZipFile(p) as z:raw=z.read('bundle_manifest.json')
        manifests.append((hashlib.sha256(raw).hexdigest(),p))
    if len({h for h,p in manifests})!=1:raise ValueError('Multiple different source versions; remove old inputs or set SOURCE_OVERRIDE')
    source=manifests[0][1]
if source.is_dir():
    manifest=json.loads((source/'bundle_manifest.json').read_text());archive=None
else:
    archive=zipfile.ZipFile(source);names=archive.namelist()
    if len(names)!=len(set(names)):raise ValueError('Duplicate source entries')
    manifest=json.loads(archive.read('bundle_manifest.json'))
    if set(names)!=set(manifest['files'])|{'bundle_manifest.json'}:raise ValueError('Source inventory mismatch')
try:
    if manifest['schema']!='hope_recovery_stage1_source_v1':raise ValueError('Wrong source schema')
    if manifest.get('origin',{}).get('delivery_revision')!='cadic-cache-r2':raise ValueError('Attach NEW r2 source; old timeout bundle is not compatible')
    for name,expected in manifest['files'].items():
        target=safe(REPO,name)
        if archive is None:
            original=safe(source,name)
            if original.is_symlink():raise ValueError('Source symlink')
            raw=original.read_bytes()
        else:raw=archive.read(name)
        if hashlib.sha256(raw).hexdigest()!=expected:raise ValueError('Source checksum mismatch: '+name)
        if target.exists() and sha(target)!=expected:raise ValueError('Working source differs; restart session: '+name)
        target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(raw)
finally:
    if archive is not None:archive.close()
(REPO/'bundle_manifest.json').write_text(json.dumps(manifest))
os.chdir(REPO);sys.path.insert(0,str(REPO))
print('[BOOTSTRAP] verified',len(manifest['files']),'source files; origin:',manifest['origin'],flush=True)
print('Phase:',PHASE,'requested controls:',JOBS,'session budget:',SESSION_HOURS,'hours',flush=True)


In [ ]:
from notebook_support.recovery_stage1.entry import run_notebook
RESULT=run_notebook(dict(phase=PHASE,inputs=INPUT,work=WORK,w6=W6,assets=ASSETS,output=OUTPUT,
    started=STARTED,session_hours=SESSION_HOURS,jobs=JOBS,device='cuda',
    W6_OVERRIDE=W6_OVERRIDE,W72_OVERRIDE=W72_OVERRIDE,RESUME_OVERRIDE=RESUME_OVERRIDE,
    MVTEC_OVERRIDE=MVTEC_OVERRIDE,CHECKPOINT_OVERRIDE=CHECKPOINT_OVERRIDE))


## Keep the saved version output

Download `hope_cad_recovery_stage1_evidence.zip` and log, or attach this version's Output to the next run.
Check `handoff.json`: `recovery_preflight_passed`, `paused_at_atomic_boundary`, or `recovery_stage1_complete_pending_review`.
A failure also exports a ZIP before the cell raises. Full scope/instructions: `docs/main_spec/recovery_stage1_handoff.md`.
